# Build a Text Classifier - Step by Step (runnable)

**How to run this notebook**
1. Open this file in **VS Code**.
2. Top-right, click **Select Kernel** -> choose the project's **.venv** Python.
3. Run each cell with **Shift + Enter**, top to bottom. Read the notes above each cell.

We will build a **model** that reads clinical text and predicts its **type**
(Physician Note / Discharge Summary / Lab Report).

> This runs **locally** (VS Code / Jupyter). Google Colab is only needed later
> for heavy GPU fine-tuning (LoRA).

## What is a 'model' and 'learning the mapping X -> y'?

- **X** = the inputs (text turned into numbers = **features**).
- **y** = the answers we want (the note **type** = **label**).
- A **model** = a rule/function that maps **X -> y**.

We do NOT write the rule by hand. The model **learns** it from labeled examples,
then predicts y for **new** X it has never seen.

In [1]:
# Cell 1 - import our tools and load the embedding model (downloads once).
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Ready.")

c:\Users\MMS\Documents\Personal AI projects\ai-ml-projects\tenet-clinical-ai\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready.


## Step 1 - Training data (features & labels)

Each example is a piece of text **plus** its correct type (the **label**).
This is **supervised learning**: we learn from labeled examples.

In [2]:
# Cell 2 - our labeled training examples: (text, label)
training = [
    ("The patient presents with hypertension. BP 150/95.",        "Physician Note"),
    ("Assessment: type 2 diabetes on metformin. Plan: continue.",  "Physician Note"),
    ("Chief complaint: headaches. Start medication, follow up.",   "Physician Note"),
    ("Physical exam unremarkable. Reports chest discomfort.",      "Physician Note"),
    ("Plan: reduce salt, exercise, recheck in four weeks.",        "Physician Note"),
    ("Follow-up care: finish antibiotics, see doctor in a week.",  "Discharge Summary"),
    ("Hospital course: pneumonia, IV antibiotics, improved.",      "Discharge Summary"),
    ("Discharge instructions: return to ER if fever returns.",     "Discharge Summary"),
    ("Patient discharged in stable condition after three days.",   "Discharge Summary"),
    ("Reason for admission: pneumonia. Discharged home.",          "Discharge Summary"),
    ("Fasting glucose 138 high. Cholesterol 220 borderline.",      "Lab Report"),
    ("Results: LDL 145, HDL 40. Repeat testing advised.",          "Lab Report"),
    ("Metabolic panel: elevated glucose. Lipid profile abnormal.", "Lab Report"),
    ("Hemoglobin A1c 7.8 percent, above target.",                  "Lab Report"),
    ("Panel: sodium 140, potassium 4.2, creatinine normal.",       "Lab Report"),
]
texts  = [t for t, label in training]
labels = [label for t, label in training]
print("We have", len(texts), "labeled examples.")

We have 15 labeled examples.


## Step 2 - Features (X): turn text into numbers

A model cannot read words - only numbers. We use the **embedding model** to turn
each text into a vector of 384 numbers (its meaning). These numbers are the
**features (X)**.

In [8]:
# Cell 3 - build X (features) and y (labels)
X = embedder.encode(texts, normalize_embeddings=True)
y = labels
print("X shape:", X.shape, "->", X.shape[0], "examples, each", X.shape[1], "numbers")
print("y (first 3 labels):", y[:3])
print("X shape:", X.shape, "->", X.shape[0], "examples, each", X.shape[1], "numbers")
print(X.shape[1])

X shape: (15, 384) -> 15 examples, each 384 numbers
y (first 3 labels): ['Physician Note', 'Physician Note', 'Physician Note']
X shape: (15, 384) -> 15 examples, each 384 numbers
384


## Why Logistic Regression?

A simple, fast, reliable **classifier** that outputs a **probability per class**.
(Despite 'regression' in the name, it does classification.) It is a strong
**baseline** - always try the simple thing first.

## Step 3 - Train the model (adjust it to the data)

`model.fit(X, y)` is the **learning**: the model starts with random internal
numbers (**weights**), then repeatedly predicts, measures its error (**loss**),
and nudges the weights to be less wrong - many times - until predictions match
the answers.

In [9]:
# Cell 4 - create and TRAIN (fit) the model
model = LogisticRegression(max_iter=1000)
model.fit(X, y)                      # <-- this is the learning
print("Trained. Classes it knows:", list(model.classes_))

Trained. Classes it knows: [np.str_('Discharge Summary'), np.str_('Lab Report'), np.str_('Physician Note')]


## What is Softmax?

The model outputs raw scores (**logits**) for each class. **Softmax** turns them
into **probabilities** between 0 and 1 that **add up to 1**. The highest one is
the **prediction**; its value is the **confidence**. `predict_proba()` gives them.

In [10]:
# Cell 5 - predict the type of NEW text (never seen in training)
def classify(text):
    x = embedder.encode([text], normalize_embeddings=True)
    label = model.predict(x)[0]
    proba = model.predict_proba(x)[0]
    probs = {c: round(float(p), 2) for c, p in zip(model.classes_, proba)}
    return label, probs

for t in [
    "Blood pressure 160/100, start medication, follow up.",
    "Discharge home with antibiotics, return if worse.",
    "Glucose 145 high, cholesterol elevated, repeat labs.",
]:
    label, probs = classify(t)
    print(t)
    print("   PREDICTED:", label, "| probabilities (softmax):", probs)
    print()

Blood pressure 160/100, start medication, follow up.
   PREDICTED: Physician Note | probabilities (softmax): {np.str_('Discharge Summary'): 0.3, np.str_('Lab Report'): 0.31, np.str_('Physician Note'): 0.38}

Discharge home with antibiotics, return if worse.
   PREDICTED: Discharge Summary | probabilities (softmax): {np.str_('Discharge Summary'): 0.5, np.str_('Lab Report'): 0.22, np.str_('Physician Note'): 0.28}

Glucose 145 high, cholesterol elevated, repeat labs.
   PREDICTED: Lab Report | probabilities (softmax): {np.str_('Discharge Summary'): 0.23, np.str_('Lab Report'): 0.48, np.str_('Physician Note'): 0.29}



## Step 4 - See softmax with your own eyes

Softmax is just math: exponentiate the scores and divide by their sum so they
add up to 1.

In [11]:
# Cell 6 - softmax by hand
def softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()

raw = np.array([2.0, 0.5, -1.0])
print("raw scores (logits):", raw)
print("after softmax       :", np.round(softmax(raw), 3), "-> sums to", round(float(softmax(raw).sum()), 3))

raw scores (logits): [ 2.   0.5 -1. ]
after softmax       : [0.786 0.175 0.039] -> sums to 1.0


## Step 5 - Overfitting vs Underfitting

We hold back some examples as a **test set** the model never trains on, to check
it works on **new** data.

- **Overfitting**: high on TRAIN, low on TEST -> memorized, did not generalize.
- **Underfitting**: low on BOTH -> too simple / not enough signal.
- **Good fit**: high on both, small gap.

Run the next cell and compare the two accuracies.

In [12]:
# Cell 7 - train/test split and compare accuracies
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y)

m = LogisticRegression(max_iter=1000).fit(X_train, y_train)
train_acc = accuracy_score(y_train, m.predict(X_train))
test_acc  = accuracy_score(y_test,  m.predict(X_test))

print("Accuracy on TRAINING data:", round(train_acc, 2))
print("Accuracy on TEST data    :", round(test_acc, 2))
print()
print("Big gap (high train, low test) => OVERFITTING")
print("Both low                        => UNDERFITTING")
print("With only 15 tiny examples, expect a low test score - more data helps.")

Accuracy on TRAINING data: 1.0
Accuracy on TEST data    : 0.4

Big gap (high train, low test) => OVERFITTING
Both low                        => UNDERFITTING
With only 15 tiny examples, expect a low test score - more data helps.


## Summary - terminology you just used

- **Features (X)** = embeddings; **Labels (y)** = note types.
- **Model** = Logistic Regression that learns X -> y.
- **Training / fitting** = adjusting weights to reduce error (loss).
- **Inference / predict** = using the trained model on new text.
- **Softmax / predict_proba** = class probabilities that sum to 1.
- **Train/test split, accuracy** = measuring generalization.
- **Overfitting / underfitting** = memorizing vs too-simple.

You just built and evaluated a real ML classifier. Change the training examples
or add a new class and re-run to see what happens!